In [ ]:
%pip install pandas openpyxl scipy

In [ ]:
from pathlib import Path
import pandas as pd

base = Path.cwd()
print("当前文件夹：", base)
print("文件夹里的 Excel：", [p.name for p in base.glob("*.xlsx")])

In [ ]:
a = pd.read_excel(base / "Model A (replication).xlsx")
b = pd.read_excel(base / "Model B (ours).xlsx")

print("Model A 行数、列数：", a.shape)
print("Model B 行数、列数：", b.shape)

print("\n院内死亡分组人数（0=存活，1=死亡）：")
print(a["died_in_hospital"].value_counts().sort_index())

In [ ]:
df = a.copy()

# female：0=男性，1=女性
df["male"] = 1 - df["female"]

# 按院内死亡结局分组
groups = {
    "Total (n=1420)": df,
    "Survival (n=1126)": df[df["died_in_hospital"] == 0],
    "Non-Survival (n=294)": df[df["died_in_hospital"] == 1],
}

rows = []

for label, field in [
    ("Age, mean (SD)", "age_at_admission"),
    ("Male, n (%)", "male"),
    ("White including subcategories, n (%)", "race_white"),
]:
    row = {"Variable": label}

    for group_name, data in groups.items():
        values = data[field].dropna()

        if field == "age_at_admission":
            row[group_name] = (
                f"{values.mean():.2f} ({values.std():.2f})"
            )
        else:
            count = int(values.sum())
            percent = count / len(values) * 100
            row[group_name] = f"{count} ({percent:.1f}%)"

    rows.append(row)

table1_preview = pd.DataFrame(rows)
display(table1_preview)

In [ ]:
# 添加检验指标，暂按均值（标准差）显示
lab_fields = {
    "INR": "mean_inr",
    "BUN": "mean_bun",
    "Potassium": "mean_potassium",
    "Sodium": "mean_sodium",
    "Creatinine": "mean_creatinine",
    "Total bilirubin": "mean_bilirubin_total",
    "ALT": "mean_alt",
    "ALP": "mean_alp",
    "AST": "mean_ast",
    "LDH": "mean_ldh",
    "CK": "mean_ck",
    "PT": "mean_pt",
    "WBC": "mean_wbc",
    "Platelet": "mean_platelet",
    "RDW": "mean_rdw",
    "Hemoglobin": "mean_hemoglobin",
    "SO2": "mean_so2",
    "PO2": "mean_po2",
    "pH": "mean_ph",
    "PTT": "mean_ptt",
    "Base excess": "mean_base_excess",
    "Albumin": "mean_albumin",
    "Chloride": "mean_chloride",
    "Anion gap": "mean_anion_gap",
    "Bicarbonate": "mean_bicarbonate",
}

lab_rows = []

for label, field in lab_fields.items():
    row = {"Variable": label}

    for group_name, data in groups.items():
        values = data[field].dropna()
        row[group_name] = (
            f"{values.mean():.2f} ({values.std():.2f})"
            if len(values) > 0 else "NA"
        )

    missing = int(df[field].isna().sum())
    row["Missing, n (%)"] = (
        f"{missing} ({missing / len(df) * 100:.1f}%)"
    )
    lab_rows.append(row)

table1_labs = pd.DataFrame(lab_rows)
display(table1_labs)

In [ ]:
continuous_fields = {
    "Weight": "weight_admit_kg",
    "Heart rate": "mean_heart_rate",
    "SBP": "mean_sbp",
    "DBP": "mean_dbp",
    "Mean arterial pressure": "mean_map",
    "Respiratory rate": "mean_resp_rate",
    "Temperature": "mean_temp_c",
    "SpO2": "mean_spo2",
    "Glucose": "mean_glucose",
    "SOFA day 1": "sofa_day1",
    "SAPS II": "sapsii",
    "Minimum GCS": "gcs_total_min",
    "Charlson index": "charlson_comorbidity_index",
}

binary_fields = {
    "Prior CHF": "prior_chf",
    "Prior COPD": "prior_copd",
    "Prior CKD": "prior_ckd",
    "Prior diabetes": "prior_diabetes",
    "Prior liver disease": "prior_liver_disease",
    "Invasive ventilation by t0": "invasive_vent_by_t0",
    "CRRT by t0": "crrt_by_t0",
    "Vasopressin": "on_vasopressin",
    "Epinephrine": "on_epinephrine",
    "Norepinephrine": "on_norepinephrine",
    "Dopamine": "on_dopamine",
}

other_rows = []

for is_binary, fields in [
    (False, continuous_fields),
    (True, binary_fields),
]:
    for label, field in fields.items():
        row = {"Variable": label}

        for group_name, data in groups.items():
            values = data[field].dropna()

            if len(values) == 0:
                result = "NA"
            elif is_binary:
                count = int(values.sum())
                result = (
                    f"{count} ({count / len(values) * 100:.1f}%)"
                )
            else:
                result = (
                    f"{values.mean():.2f} ({values.std():.2f})"
                )

            row[group_name] = result

        missing = int(df[field].isna().sum())
        row["Missing, n (%)"] = (
            f"{missing} ({missing / len(df) * 100:.1f}%)"
        )
        other_rows.append(row)

table1_other = pd.DataFrame(other_rows)
display(table1_other)

In [ ]:
import numpy as np
from scipy import stats

# 汇总之前已经使用的字段
all_continuous = {
    "Age, mean (SD)": "age_at_admission",
    **lab_fields,
    **continuous_fields,
}

all_binary = {
    "Male, n (%)": "male",
    "White including subcategories, n (%)": "race_white",
    **binary_fields,
}

# 合并已有的三部分
table1 = pd.concat(
    [table1_preview, table1_labs, table1_other],
    ignore_index=True
)

# 用变量名称匹配字段，避免错位
field_map = {**all_continuous, **all_binary}

def format_p(p):
    if pd.isna(p):
        return "NA"
    return "<0.001" if p < 0.001 else f"{p:.3f}"

p_values = []
test_names = []
missing_total = []
missing_survival = []
missing_death = []

survival = df[df["died_in_hospital"] == 0]
death = df[df["died_in_hospital"] == 1]

def missing_text(data, field):
    n = int(data[field].isna().sum())
    return f"{n} ({n / len(data) * 100:.1f}%)"

for label in table1["Variable"]:
    field = field_map[label]
    x = survival[field].dropna()
    y = death[field].dropna()

    p = np.nan
    test = "Not tested"

    if len(x) > 0 and len(y) > 0:
        if label in all_binary:
            assert pd.concat([x, y]).isin([0, 1]).all()

            observed = np.array([
                [(x == 0).sum(), (x == 1).sum()],
                [(y == 0).sum(), (y == 1).sum()],
            ])

            if (observed.sum(axis=0) == 0).any():
                test = "Constant; not tested"
            else:
                chi = stats.chi2_contingency(
                    observed, correction=False
                )

                if (chi.expected_freq < 5).any():
                    p = stats.fisher_exact(observed).pvalue
                    test = "Fisher exact"
                else:
                    p = chi.pvalue
                    test = "Pearson chi-square"
        else:
            if pd.concat([x, y]).nunique() == 1:
                test = "Constant; not tested"
            else:
                p = stats.mannwhitneyu(
                    x, y,
                    alternative="two-sided",
                    method="auto"
                ).pvalue
                test = "Mann-Whitney U"

    p_values.append(format_p(p))
    test_names.append(test)
    missing_total.append(missing_text(df, field))
    missing_survival.append(missing_text(survival, field))
    missing_death.append(missing_text(death, field))

table1["Missing, n (%)"] = missing_total
table1["Missing: Survival"] = missing_survival
table1["Missing: Non-Survival"] = missing_death
table1["P-value"] = p_values
table1["Test"] = test_names

# 显示所有行
with pd.option_context("display.max_rows", None):
    display(table1)

In [ ]:
print("变量总数：", len(table1))
print("重复变量数：", table1["Variable"].duplicated().sum())
print("没有完成检验的变量：")
display(
    table1.loc[
        table1["P-value"] == "NA",
        ["Variable", "P-value", "Test"]
    ]
)

In [ ]:
output_path = base / "Table1_draft.xlsx"

if output_path.exists():
    print("同名文件已存在，请更换文件名后再保存。")
else:
    table1.to_excel(output_path, index=False)
    print("已保存到：", output_path)